In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential


In [5]:
batch_size = 32
img_height = 180    
img_width = 180
epochs = 100

In [29]:
from PIL import Image
import os

bad_files = []

d_path = "C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease"

# Walk through the directory and check each image

for root, dirs, files in os.walk(path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(path)
            img.verify()  # check corruption
        except:
            bad_files.append(path)

print("Bad files:", bad_files)

#  delete bad files
# for f in bad_files:
    # os.remove(f)

Bad files: ['C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease', 'C:\\Users\\HUSSAINI

In [30]:
path = "C:\\Users\\HUSSAINI IBRAHIM\\Documents\\agrigani\\backend\\ml_service\\plant_disease"

train_ds = tf.keras.utils.image_dataset_from_directory(
  path,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

Found 25156 files belonging to 22 classes.
Using 20125 files for training.


In [31]:
test_ds = tf.keras.utils.image_dataset_from_directory(
  path,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 25156 files belonging to 22 classes.
Using 5031 files for validation.


In [32]:
# Configure the dataset for performance
ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
ds_test = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [33]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])


In [34]:
from tensorflow.keras.applications import MobileNetV2
base_model = MobileNetV2(input_shape=(img_height, img_width, 3),
                         include_top=False,
                         weights='imagenet')

base_model.trainable = False

C:\Users\HUSSAINI IBRAHIM\AppData\Local\Temp\ipykernel_16260\2317781121.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(img_height, img_width, 3),


In [35]:
num_l = len(train_ds.class_names)

model = Sequential(
    [
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_l, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_l, activation="relu")
        
    ]
)

In [36]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])


In [37]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 6, 6, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [38]:
history = model.fit(
  train_ds,
  validation_data=test_ds,
  epochs= 50 )

Epoch 1/50
 11/629 ━━━━━━━━━━━━━━━━━━━━ 3:14 315ms/step - accuracy: 0.0450 - loss: 3.2458

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
jpeg::Uncompress failed. Invalid JPEG data or crop window.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_18834]